# Production Multi-Agent Salary Intelligence System
This notebook demonstrates a production-style multi-agent architecture with:
- Tool usage
- Async orchestration
- Logging
- Retry logic
- Fine-tuned model inference


In [ ]:
!pip install tenacity

In [ ]:
import asyncio
import logging
from typing import Dict
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

client = OpenAI()

MODEL = 'ft:gpt-4.1-nano-2025-04-14:personal:salary-predictor:DH8pG4fE'

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('multi-agent')

## Base Agent Class

In [ ]:
class Agent:

    def __init__(self, name, system_prompt):
        self.name = name
        self.system_prompt = system_prompt

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    async def run(self, input_text):

        logger.info(f'Agent {self.name} running...')

        response = client.chat.completions.create(
            model='gpt-4.1-nano',
            temperature=0,
            messages=[
                {'role':'system','content':self.system_prompt},
                {'role':'user','content':input_text}
            ]
        )

        return response.choices[0].message.content

## Tool: Market Salary Estimator

In [ ]:
def salary_market_tool(role, experience):

    base = 100000

    if 'senior' in role.lower():
        base += 60000

    if 'ml' in role.lower() or 'ai' in role.lower():
        base += 30000

    try:
        years = int(experience.split()[0])
        base += years * 5000
    except:
        pass

    return f'Estimated market salary around ${base}'

## Job Parser Agent

In [ ]:
job_parser = Agent(
    'Job Parser',
    '''Extract structured job data:
    - title
    - experience
    - skills
    - location'''
)

## Market Analyst Agent

In [ ]:
market_agent = Agent(
    'Market Analyst',
    '''Analyze the job and estimate salary ranges.'''
)

## Salary Prediction Agent (Fine-tuned model)

In [ ]:
def salary_predictor(job_description):

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{'role':'user','content':job_description}]
    )

    return response.choices[0].message.content

## Decision Agent

In [ ]:
decision_agent = Agent(
    'Decision Agent',
    '''Combine agent outputs and produce a final salary decision.'''
)

## Multi-Agent Orchestrator

In [ ]:
async def run_multi_agent_pipeline(job_description: str) -> Dict:

    parsed_job = await job_parser.run(job_description)

    market_analysis = await market_agent.run(parsed_job)

    salary_prediction = salary_predictor(job_description)

    final_decision = await decision_agent.run(
        f"""
        Parsed Job:
        {parsed_job}

        Market Analysis:
        {market_analysis}

        Model Prediction:
        {salary_prediction}
        """
    )

    return {
        'parsed_job': parsed_job,
        'market_analysis': market_analysis,
        'salary_prediction': salary_prediction,
        'final_decision': final_decision
    }

## Example Job

In [ ]:
job_description = '''
Job Title: Senior Machine Learning Engineer
Company: Google
Location: Mountain View
Experience: 6 years
Skills: Python, ML, Deep Learning
'''

## Run System

In [ ]:
result = await run_multi_agent_pipeline(job_description)

for k,v in result.items():
    print(f"\n--- {k.upper()} ---\n")
    print(v)